# Milestone 2 — RAG Pipeline Exploration

This notebook walks through the full RAG pipeline:
1. Load Milestone 1 retrievers from disk.
2. Wrap them as LangChain `BaseRetriever` subclasses.
3. Build the context from retrieved documents.
4. Compare the three prompt variants on the same query.
5. Run end-to-end RAG over 10 evaluation queries.

**Prereqs:** `make setup` has been run, `HF_TOKEN` is set in `.env`, and the token has accepted the Meta Llama 3 license.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

True

In [2]:
from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

INDICES = Path.cwd().parent / "indices"

bm25 = BM25Retriever()
bm25.load(str(INDICES / "bm25_index.pkl"))

semantic = SemanticRetriever()
semantic.load(str(INDICES / "faiss_index"))

print(f"BM25 corpus: {len(bm25.corpus):,} products")
print(f"Semantic corpus: {len(semantic.corpus):,} products")

/Users/williamchong/miniforge3/envs/575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7199.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BM25 corpus: 112,590 products
Semantic corpus: 112,590 products


In [3]:
from src.retrievers_lc import wrap_retriever

bm25_lc = wrap_retriever("BM25", bm25, semantic, top_k=5)
semantic_lc = wrap_retriever("Semantic", bm25, semantic, top_k=5)
hybrid_lc = wrap_retriever("Hybrid", bm25, semantic, top_k=5)

query = "vitamin c serum for dark spots"
for name, r in [("BM25", bm25_lc), ("Semantic", semantic_lc), ("Hybrid (RRF)", hybrid_lc)]:
    docs = r.invoke(query)
    print(name + ":")
    for d in docs[:3]:
        asin = d.metadata["parent_asin"]
        title = d.metadata["title"]
        print(f"  [{asin}] {title}")
    print()

BM25:
  [B00VQHFBBE] Vitamin C Serum for Face - Best 20% Vitamin C + E + Hyaluronic Acid – Anti Aging Natural Skin Care for Dark Spots, Wrinkles, Sun Spots, Fine Lines, Hyperpigmentation, Age Spots, Sun Damage – Vitamin C Serum for Eye Area, Under Eyes Dark Circles, Skin Lightening – 1 Oz – 100%
  [B019D6WNJM] Vitamin C Serum - 10% Pure Vitamin C, Ascorbic Acid, Liposomal Technology, Antioxidant, Ultimate Collagen Booster, Brightener, Fade Dark Spots, Exfoliate, Even Tone/Texture (1 Oz / 30ml)
  [B07PC2CNK7] Eva St. Claire Vitamin C Serum. Anti-aging brightening serum with Vitamin C, Ferulic Acid, Collagen, Aloe Vera for dark spots, skin discoloration. 1 fl oz (30ml)

Semantic:
  [B01EVFIHW2] Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark Underarms & Under Eye Dark Circles. Vitamin C Collagen Boost for Smoother Complexion
  [B07F7G3PJW] Majestic Pure Serum with Hyaluronic Acid for Face & Skin - Moisturizing and Serum - Redu

In [4]:
from src.prompts import build_context

docs = hybrid_lc.invoke(query)
print(build_context(docs))

[1] ASIN: B07PC2CNK7 | Title: Eva St. Claire Vitamin C Serum. Anti-aging brightening serum with Vitamin C, Ferulic Acid, Collagen, Aloe Vera for dark spots, skin discoloration. 1 fl oz (30ml) | Rating: 3.6/5 | Price: $nan
Review/Description: Eva St. Claire Vitamin C Serum. Anti-aging brightening serum with Vitamin C, Ferulic Acid, Collagen, Aloe Vera for dark spots, skin discoloration. 1 fl oz (30ml) This serum is very light weight and non greasy. It makes your face feel soft and subtle. I would definitely purchase again.

[2] ASIN: B01EVFIHW2 | Title: Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark Underarms & Under Eye Dark Circles. Vitamin C Collagen Boost for Smoother Complexion | Rating: 3.0/5 | Price: $nan
Review/Description: Dark Spot Reducing Serum: Visibly Reduces Density of Age Spots, Sun Spots & Liver Spots. Fades Freckles, Dark Underarms & Under Eye Dark Circles. Vitamin C Collagen Boost for Smoother Complexion I

## Prompt Variant Comparison

Same query, three system prompts. We expect:
- `strict_citation`: short, ASIN-cited, may say "I don't have enough information."
- `helpful_shopper`: more conversational, may include price/rating commentary.
- `structured_json`: JSON object with `recommendation`, `reasoning`, `asins`.

In [5]:
from src.rag_pipeline import RAGPipeline

comparison_query = "what's a good vitamin C serum for dark spots under $30?"

for variant in ["strict_citation", "helpful_shopper", "structured_json"]:
    pipeline = RAGPipeline(
        bm25=bm25,
        semantic=semantic,
        retriever_name="Hybrid",
        prompt_name=variant,
        top_k=5,
    )
    result = pipeline.answer(comparison_query)
    print("=== " + variant + " ===")
    print(result["answer"])
    print()

=== strict_citation ===
Based on the provided reviews and metadata, I would recommend the following options for a Vitamin C serum for dark spots under $30:

1. Eva St. Claire Vitamin C Serum (ASIN: B07PC2CNK7) - Although the price is not explicitly mentioned, it's an affordable option that works for dark spots, and the reviewer mentions they would "definitely purchase again."
2. Vitamin C Serum for Face (ASIN: B00VQHFBBE) - Although the price is not explicitly mentioned, it's a 3.3-star rated product that contains 20% Vitamin C, which might be effective for dark spots. However, be aware that it did not satisfy all reviewers.

Please note that the prices are not explicitly mentioned in the provided reviews. If you need a more accurate price, I would recommend checking the prices on Amazon for these products.

=== helpful_shopper ===
Based on the provided reviews, a good vitamin C serum for dark spots under $30 can be the "Vitamin C Serum" with ASIN B01AISLTUI (Essy Advanced vitamin C se

## Evaluation Run (10 Queries)

Run the default pipeline (Hybrid + strict_citation + top_k=5) over the 10 RAG queries.
Outputs go to `data/eval_outputs/rag_eval.json` for hand-rating in `results/milestone2_discussion.md`.

In [6]:
import json
import pandas as pd

EVAL_OUT = Path.cwd().parent / "data" / "eval_outputs" / "rag_eval.json"
EVAL_OUT.parent.mkdir(parents=True, exist_ok=True)

queries_df = pd.read_csv(Path.cwd().parent / "data" / "processed" / "rag_queries.csv")

default_pipeline = RAGPipeline(
    bm25=bm25,
    semantic=semantic,
    retriever_name="Hybrid",
    prompt_name="strict_citation",
    top_k=5,
)

results = []
for _, row in queries_df.iterrows():
    qid = row["query_id"]
    q = row["query"]
    print(f"Query {qid}: {q}")
    answer = default_pipeline.answer(q)
    results.append({
        "query_id": int(qid),
        "query": q,
        "category": row["category"],
        "answer": answer["answer"],
        "sources": answer["sources"],
    })

EVAL_OUT.write_text(json.dumps(results, indent=2))
print(f"Saved {len(results)} eval outputs to {EVAL_OUT}")

Query 1: vitamin C serum for brightening
Query 2: sunscreen with SPF 50
Query 3: coconut oil shampoo for dry hair
Query 4: what helps with mild acne and post-acne marks?
Query 5: gentle face wash that won't irritate sensitive skin
Query 6: something to make my hair look less frizzy in humidity
Query 7: affordable retinol treatment for fine lines
Query 8: best skincare routine for oily acne-prone teenage skin under $30
Query 9: cruelty-free moisturizer good for winter on sensitive skin
Query 10: what helps with sun damage on fair skin around the eyes
Saved 10 eval outputs to /Users/williamchong/Documents/UBC_MDS/Block6/575advml/DSCI_575_project_willchh_jiromig/data/eval_outputs/rag_eval.json
